# Prueba rápida: modelo de Hugging Face **remoto** (Inference API)

Notebook mínima para llamar modelos de Hugging Face **sin descargarlos**: la inferencia corre en los servidores de Hugging Face y acá solo mandamos la petición y recibimos la respuesta (como cuando llamás a la API de OpenAI/Anthropic).

Diferencias clave con `prueba_modelo_huggingface.ipynb` (la versión local):

| | Local (`pipeline`) | Remota (`InferenceClient`) |
|---|---|---|
| Descarga el modelo | Sí (cientos de MB, una vez) | No |
| Necesita GPU/CPU potente | Puede ser lento en CPU | No, corre en la nube |
| Necesita internet | Solo la primera vez | Siempre (cada llamada) |
| Necesita token de HF | No (modelos abiertos) | **Sí** (gratis) |
| Latencia | Rápida tras cargar | Depende de la red + "cold start" del modelo |


## 0. Conseguir un token de Hugging Face (gratis)

1. Creá una cuenta en [huggingface.co](https://huggingface.co) si no tenés.
2. Andá a **Settings → Access Tokens** (o directamente [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)).
3. Creá un token nuevo con permiso de lectura (`Read`) — alcanza para la Inference API.
4. **No lo pegues directo en una celda ni lo subas a git.** Lo mejor es setearlo como variable de entorno antes de abrir Jupyter:
   - PowerShell: `$env:HF_TOKEN = "hf_..."`
   - Bash: `export HF_TOKEN="hf_..."`

   Si no lo seteaste como variable de entorno, la celda de abajo te lo va a pedir con `getpass` (no queda visible en pantalla ni se guarda en el notebook).

## Instalación de dependencias

En local ya está en `requirements.txt`. En Colab, descomentá y corré la celda siguiente.

In [1]:
# !pip install -q huggingface_hub

## Imports y autenticación

In [2]:
import os
from getpass import getpass

from dotenv import load_dotenv
from huggingface_hub import InferenceClient

# Orden de busqueda del token:
# 1) variable de entorno HF_TOKEN ya seteada
# 2) archivo .env en esta carpeta (ver .env.example) -> lo carga load_dotenv()
# 3) si no hay ninguna, lo pide de forma segura (no se muestra en pantalla ni se guarda)
load_dotenv()
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Pega tu HF token (no se va a mostrar): ")

# provider="hf-inference" fuerza el servicio gratuito original de Hugging Face.
# Sin esto, el ruteo automatico de "providers" puede fallar con modelos viejos como gpt2.
client = InferenceClient(provider="hf-inference", token=HF_TOKEN)
print("Cliente listo. Las llamadas de aca en adelante van a los servidores de Hugging Face.")


Cliente listo. Las llamadas de aca en adelante van a los servidores de Hugging Face.


## 1. Question answering remoto (`deepset/roberta-base-squad2`)

Nota: la Inference API gratuita de Hugging Face (`hf-inference`) ya **no sirve modelos de generación de texto libre** como `gpt2` (los movieron a proveedores de pago con facturacion aparte). Para una demo 100% gratuita usamos **question answering extractivo**: le damos un contexto y una pregunta, y el modelo extrae la respuesta del texto (sin generar nada nuevo).

In [3]:
contexto = (
    "Machine learning is a subfield of artificial intelligence that focuses on building "
    "systems that learn from data. It has become central to many modern applications, "
    "from recommendation systems to voice assistants."
)
pregunta = "What is machine learning a subfield of?"

try:
    resultado = client.question_answering(
        question=pregunta,
        context=contexto,
        model="deepset/roberta-base-squad2",
    )
    print(f"Pregunta: {pregunta}")
    print(f"Respuesta: {resultado.answer}  (score={resultado.score:.4f})")
except Exception as e:
    print("No se pudo llamar a la Inference API:", e)
    print("Chequea que HF_TOKEN sea valido y que tengas conexion a internet.")


Pregunta: What is machine learning a subfield of?
Respuesta: artificial intelligence  (score=0.9860)


## 2. Relleno de máscara remoto (`distilbert-base-uncased`)

Mismo modelo que en la Sección 2 del notebook local, pero vía API.

In [4]:
frase = "The capital of France is [MASK]."

try:
    predicciones = client.fill_mask(frase, model="distilbert/distilbert-base-uncased")
    print(f"Frase: {frase}\n")
    for p in predicciones[:5]:
        print(f"  {p['token_str']:12s} score={p['score']:.4f}")
except Exception as e:
    print("No se pudo llamar a la Inference API:", e)


Frase: The capital of France is [MASK].

  marseille    score=0.1427
  nantes       score=0.0902
  toulouse     score=0.0881
  paris        score=0.0862
  lyon         score=0.0772


## 3. Ejercicio

1. Cambiá el `contexto` y la `pregunta` de la Sección 1, y la `frase` de la Sección 2, y compará las respuestas.
2. Cronometrá una llamada remota (`%%time` al principio de la celda) y compará contra el tiempo de inferencia local del otro notebook. ¿Cuál es más rápido en tu maquina? ¿Por qué creés que pasa eso?
3. Probá apagar el WiFi/datos y volver a correr una celda de la Sección 1 o 2. ¿Qué error obtenés? Compará con qué pasaría en el notebook local (que ya tiene el modelo descargado).

Si todo corrió sin errores, ya sabés llamar modelos de Hugging Face **sin** descargarlos localmente. ✅